# Fine-tuning BERT, RoBERT or XLM-RoBERTa for multi-label text classification

## Set-up environment

First, we install the libraries which we'll use: HuggingFace Transformers and Datasets.

In [ ]:
#!pip install -U transformers
!pip install transformers==4.42

In [ ]:
!pip install -q datasets

## Load dataset

We load REDv2 / REDv2_EN / REDv2_EN_ANN, multi-label text classification datasets from the [hub](https://huggingface.co/).





In [ ]:
from datasets import load_dataset

dataset = load_dataset("Alegzandra/REDv2_EN")

As we can see, the dataset contains 3 splits: one for training, one for validation and one for testing.

In [ ]:
dataset

We remove columns that we don't need:

In [ ]:
dataset = dataset.remove_columns(['agreed_labels', 'annotator1', 'annotator2', 'annotator3', 'percentage_labels', 'sum_labels', ])

In [ ]:
dataset

Let's check the first example of the training split:

In [ ]:
example = dataset['train'][:10]
print(type(example))
print(example)

We replace anonimizations with words, so that text vectorization makes sense:

We change "Person" into "Persoana" when using RoBERT for vectorization.

In [ ]:
def prep(in_str):
  elem = in_str.replace("<|EMAIL|>","email")
  elem = elem.replace("<|TEL|>","")
  elem = elem.replace("<|USERNAME|>","")
  elem = elem.replace("<|URL|>","")
  out_str = elem.replace("<|PERSON|>","Person")
  return out_str

print(prep("<|PERSON|> s-a dus la <|EMAIL|> sa <|URL|>"))


In [ ]:
def add_prefix(example):

    example["text"] = prep(example["text"])

    return example

In [ ]:
dataset = dataset.map(add_prefix)

In [ ]:
dataset

In [ ]:
example = dataset['train'][:10]
print(type(example))
print(example)

The dataset consists of tweets, labeled with one or more emotions.

Let's create a list that contains the labels, as well as 2 dictionaries that map labels to integers and back.

In [ ]:
labels = [label for label in dataset['train'].features.keys() if label not in ['text_id', 'text']]
id2label = {idx:label for idx, label in enumerate(labels)}
label2id = {label:idx for idx, label in enumerate(labels)}
labels

## Preprocess data

As models like BERT don't expect text as direct input, but rather `input_ids`, etc., we tokenize the text using the tokenizer. Here I'm using the `AutoTokenizer` API, which will automatically load the appropriate tokenizer based on the checkpoint on the hub.

What's a bit tricky is that we also need to provide labels to the model. For multi-label text classification, this is a matrix of shape (batch_size, num_labels). Also important: this should be a tensor of floats rather than integers, otherwise PyTorch' `BCEWithLogitsLoss` (which the model will use) will complain, as explained [here](https://discuss.pytorch.org/t/multi-label-binary-classification-result-type-float-cant-be-cast-to-the-desired-output-type-long/117915/3).

**For our experiments transformer models are chosen using the following schema:**

Model 1 - `dumitrescustefan/bert-base-romanian-cased-v1`

Models 2, 4 and 6 - `xlm-roberta-base`

Models 3 and 5 - `bert-base-cased`

In [ ]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def preprocess_data(examples):
  # take a batch of texts
  text = examples["text"]
  # encode them
  encoding = tokenizer(text, padding="max_length", truncation=True, max_length=128)
  # add labels
  labels_batch = {k: examples[k] for k in examples.keys() if k in labels}
  # create numpy array of shape (batch_size, num_labels)
  labels_matrix = np.zeros((len(text), len(labels)))
  # fill numpy array
  for idx, label in enumerate(labels):
    labels_matrix[:, idx] = labels_batch[label]

  encoding["labels"] = labels_matrix.tolist()

  return encoding

In [ ]:
encoded_dataset = dataset.map(preprocess_data, batched=True, remove_columns=dataset['train'].column_names)

In [ ]:
example = encoded_dataset['train'][0]
print(example.keys())

In [ ]:
tokenizer.decode(example['input_ids'])

In [ ]:
example['labels']

In [ ]:
[id2label[idx] for idx, label in enumerate(example['labels']) if label == 1.0]

Finally, we set the format of our data to PyTorch tensors. This will turn the training, validation and test sets into standard PyTorch [datasets](https://pytorch.org/docs/stable/data.html).

In [ ]:
encoded_dataset.set_format("torch")

## Define model

Here we define a model that includes a pre-trained base (i.e. the weights from bert-base-uncased) are loaded, with a random initialized classification head (linear layer) on top. One should fine-tune this head, together with the pre-trained base on a labeled dataset.

This is also printed by the warning.

We set the `problem_type` to be "multi_label_classification", as this will make sure the appropriate loss function is used (namely [`BCEWithLogitsLoss`](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)). We also make sure the output layer has `len(labels)` output neurons, and we set the id2label and label2id mappings.

**For our experiments the transformer models:**

RoBERT - `dumitrescustefan/bert-base-romanian-cased-v1`

XLM RoBERTa - `xlm-roberta-base`

BERT - `bert-base-cased`

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base",
                                                           problem_type="multi_label_classification",
                                                           num_labels=len(labels),
                                                           id2label=id2label,
                                                           label2id=label2id)

## Train the model!

We are going to train the model using HuggingFace's Trainer API. This requires us to define 2 things:

* `TrainingArguments`, which specify training hyperparameters. All options can be found in the [docs](https://huggingface.co/transformers/main_classes/trainer.html#trainingarguments). Below, we for example specify that we want to evaluate after every epoch of training, we would like to save the model every epoch, we set the learning rate, the batch size to use for training/evaluation, how many epochs to train for, and so on.
* a `Trainer` object (docs can be found [here](https://huggingface.co/transformers/main_classes/trainer.html#id1)).

In [ ]:
batch_size = 8
metric_name = "f1"

In [ ]:
!pip install peft==0.10.0

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    f"romanian_bert_cased-finetuned-on-REDv2",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    gradient_accumulation_steps = 2,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    push_to_hub=False,
    report_to="none"
)

We are also going to compute metrics while training. For this, we need to define a `compute_metrics` function, that returns a dictionary with the desired metric values.

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch

def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, we apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = labels
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions,
            tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds,
        labels=p.label_ids)
    return result

Let's verify a batch as well as a forward pass:

In [ ]:
encoded_dataset['train'][0]['labels'].type()

In [ ]:
encoded_dataset['train']['input_ids'][0]

In [ ]:
#forward pass
outputs = model(input_ids=encoded_dataset['train']['input_ids'][0].unsqueeze(0), labels=encoded_dataset['train'][0]['labels'].unsqueeze(0))
outputs

Let's start training!

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

## Evaluate

After training, we evaluate our model on the validation set.

In [ ]:
# Save the model
trainer.save_model()

In [ ]:
trainer.evaluate(eval_dataset=encoded_dataset["test"])

In [ ]:
def predict_emotions(batch_texts, threshold=0.5):
    enc = tokenizer(batch_texts, truncation=True, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.sigmoid(logits).cpu().numpy()  # move back to CPU for numpy
    preds = (probs >= threshold).astype(int)
    return preds

In [ ]:
device="cuda"

In [ ]:
# 4. Collect results
results = []
batch_size = 16
dataset = load_dataset("Alegzandra/REDv2EN", split="test")
for i in range(0, len(dataset), batch_size):
    batch = dataset[i : i + batch_size]
    texts = batch['text']
    preds_bin = predict_emotions(texts)

    for text, pred in zip(texts, preds_bin):
        # Get list of predicted emotions
        predicted_emotions = [label for label, val in zip(labels, pred) if val == 1]
        emotions_str = ", ".join(predicted_emotions) if predicted_emotions else "None"
        results.append({"text": text, "emotions": emotions_str})

In [ ]:
import pandas as pd

In [ ]:
# 5. Save to Excel
df = pd.DataFrame(results)
output_file = "Predictions model 17.xlsx"
df.to_excel(output_file, index=False)

This notebook was created by adapting this tutorial: https://github.com/NielsRogge/Transformers-Tutorials/blob/master/BERT/Fine_tuning_BERT_(and_friends)_for_multi_label_text_classification.ipynb